# SimCLR on CIFAR-10 — Tutorial

This notebook walks through a complete self-supervised pre-training run using **SimCLR** on **CIFAR-10** with `stable-pretraining`.

You will learn to:
1. **Load data** — CIFAR-10 with two augmented views via `MultiViewTransform`
2. **Build a Module** — wire `simclr_forward` with a ResNet-18 backbone and NT-Xent loss
3. **Add callbacks** — `OnlineProbe`, `OnlineKNN`, and `RankMe` for real-time quality monitoring
4. **Train with Manager** — the recommended programmatic entry point

> **Quick demo** — this notebook runs **5 epochs** so you can see the full pipeline in under 5 minutes on CPU. For publication-quality representations, set `max_epochs=1000` with a LARS optimizer and cosine schedule (see `examples/simclr_cifar10_config.yaml`).

In [ ]:
# Uncomment to install from the repo root if not already installed
# !pip install -e ".[dev]"

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchmetrics
import lightning as pl

import stable_pretraining as spt
from stable_pretraining.data import transforms
from stable_pretraining.forward import simclr_forward

print(f"stable-pretraining {spt.__version__}")
print(f"PyTorch {torch.__version__}  |  Lightning {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1 — Data

SimCLR needs **two randomly augmented views** of every image per batch. `MultiViewTransform` wraps two augmentation pipelines and produces a list of view dicts that `simclr_forward` expects.

`FromTorchDataset` converts any PyTorch dataset with tuple outputs `(image, label)` into the `{"image": ..., "label": ...}` dict format that `stable-pretraining` uses everywhere. Data flows as dicts so that callbacks can read any intermediate value (embeddings, labels, loss) with zero wiring.

In [ ]:
# spt.data.static.CIFAR10 is {"mean": [...], "std": [...]}
CIFAR10_STATS = spt.data.static.CIFAR10

# Two augmented views for SimCLR on 32x32 CIFAR images
simclr_transform = transforms.MultiViewTransform([
    # View 1: standard crop + color jitter
    transforms.Compose(
        transforms.RGB(),
        transforms.RandomResizedCrop((32, 32), scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1, p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToImage(**CIFAR10_STATS),
    ),
    # View 2: tighter crop + solarize
    transforms.Compose(
        transforms.RGB(),
        transforms.RandomResizedCrop((32, 32), scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1, p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomSolarize(threshold=0.5, p=0.2),
        transforms.ToImage(**CIFAR10_STATS),
    ),
])

# Deterministic transform for validation (no augmentation)
val_transform = transforms.Compose(
    transforms.RGB(),
    transforms.Resize((32, 32)),
    transforms.ToImage(**CIFAR10_STATS),
)

# Wrap CIFAR-10 in dict format
train_dataset = spt.data.FromTorchDataset(
    torchvision.datasets.CIFAR10(root="./data", train=True, download=True),
    names=["image", "label"],
    transform=simclr_transform,
)
val_dataset = spt.data.FromTorchDataset(
    torchvision.datasets.CIFAR10(root="./data", train=False, download=True),
    names=["image", "label"],
    transform=val_transform,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=2, drop_last=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=256, num_workers=2
)

data = spt.data.DataModule(train=train_loader, val=val_loader)
print(f"Train: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_dataset)} samples ({len(val_loader)} batches)")

## Step 2 — Module

`spt.Module` is the `LightningModule` wrapper at the core of `stable-pretraining`. You pass it:
- a **`forward` function** — `simclr_forward` defines the full SSL computation
- any **sub-modules** the forward function needs (`backbone`, `projector`, `simclr_loss`)
- an **`optim` config** (optimizer + optional scheduler)

`training_step` and `validation_step` are built automatically. Because `simclr_forward` writes `"embedding"` and `"label"` into the output dict, every callback added later can read those keys with no changes to this code.

In [ ]:
# ResNet-18 with the classifier head removed → 512-dim embeddings
# low_resolution=True adapts the first conv and removes MaxPool for 32x32 inputs
backbone = spt.backbone.from_torchvision("resnet18", low_resolution=True)
backbone.fc = nn.Identity()

# 3-layer MLP projector (SimCLR paper architecture)
projector = nn.Sequential(
    nn.Linear(512, 2048), nn.BatchNorm1d(2048), nn.ReLU(inplace=True),
    nn.Linear(2048, 2048), nn.BatchNorm1d(2048), nn.ReLU(inplace=True),
    nn.Linear(2048, 256),
)

module = spt.Module(
    forward=simclr_forward,
    backbone=backbone,
    projector=projector,
    simclr_loss=spt.losses.NTXEntLoss(temperature=0.5),
    optim={
        # Adam for this quick demo; use LARS + LinearWarmupCosineAnnealing for full runs
        "optimizer": {"type": "Adam", "lr": 1e-3},
        "scheduler": "CosineAnnealingLR",
        "interval": "epoch",
    },
)

trainable = sum(p.numel() for p in module.parameters() if p.requires_grad) / 1e6
print(f"Trainable parameters: {trainable:.1f}M")

## Step 3 — Callbacks

Callbacks are the key observability feature of `stable-pretraining`. Because `simclr_forward` puts `"embedding"` and `"label"` into the output dict, every callback below can consume those values **without touching the forward function or training step**.

| Callback | What it monitors |
|---|---|
| `OnlineProbe` | Trains a linear classifier on frozen embeddings — tracks downstream top-1/top-5 accuracy each epoch |
| `OnlineKNN` | k-NN accuracy using a rolling embedding queue — zero gradient cost |
| `RankMe` | Effective rank of the embedding matrix via singular values — a drop signals dimensional collapse |

In [ ]:
# Linear probe: trains a single Linear(512, 10) on top of frozen backbone embeddings
linear_probe = spt.OnlineProbe(
    module,
    name="linear_probe",
    input="embedding",   # key written by simclr_forward
    target="label",      # key written by simclr_forward
    probe=nn.Linear(512, 10),
    loss=nn.CrossEntropyLoss(),
    metrics={
        "top1": torchmetrics.classification.MulticlassAccuracy(10),
        "top5": torchmetrics.classification.MulticlassAccuracy(10, top_k=5),
    },
)

# KNN probe: non-parametric; maintains a rolling queue of past embeddings
knn_probe = spt.OnlineKNN(
    name="knn",
    input="embedding",
    target="label",
    queue_length=10000,
    input_dim=512,
    k=10,
    metrics={"top1": torchmetrics.classification.MulticlassAccuracy(10)},
)

# RankMe: tracks representation rank — collapse → rank approaches 1
rankme = spt.RankMe(name="rankme", target="embedding", queue_length=10000, target_shape=512)

## Step 4 — Train

`spt.Manager` wraps `pl.Trainer` and adds:
- **Deterministic run IDs** — consistent across all ranks of the same SLURM job
- **Automatic checkpointing** to a `cache_dir` run directory (if `spt.set(cache_dir=...)` is configured)
- **SIGTERM handling** — checkpoint + requeue on SLURM preemption

Call `manager()` instead of `trainer.fit()` to get all of this for free.

In [ ]:
trainer = pl.Trainer(
    max_epochs=5,          # Increase to 1000 for publication-quality results
    accelerator="auto",   # Uses GPU if available, otherwise CPU
    logger=False,
    enable_checkpointing=False,
    callbacks=[linear_probe, knn_probe, rankme],
)

manager = spt.Manager(trainer=trainer, module=module, data=data)
manager()

## Step 5 — Evaluate

Run a final validation pass to get end-of-training metrics. After only 5 epochs, probe accuracy will be low — representations are still forming. At 1000 epochs with LARS + cosine schedule, SimCLR reaches ~74% linear top-1 on CIFAR-10.

Key metrics to watch:
- `eval/linear_probe_top1` — linear probe accuracy (primary downstream benchmark)
- `eval/knn_top1` — k-NN accuracy (no training, free signal)
- `eval/rankme` — effective rank (higher is better; collapse → approaches 1)

In [ ]:
results = manager.validate()

if results:
    print(f"{'Metric':<45} {'Value':>8}")
    print("-" * 55)
    for key, val in sorted(results[0].items()):
        print(f"{key:<45} {val:>8.4f}")

## Next steps

**Scale up training:**
```python
# In the module optim config, switch to:
optim={
    "optimizer": {"type": "LARS", "lr": 5.0, "weight_decay": 1e-6},
    "scheduler": "LinearWarmupCosineAnnealing",
    "interval": "epoch",
}
# And set max_epochs=1000 in the Trainer
```

**Switch SSL method** — replace `simclr_forward` with any of the 9 forward functions:
```python
from stable_pretraining.forward import byol_forward, vicreg_forward, barlow_twins_forward
```
See `METHODS.md` for the full catalog and required attributes for each method.

**Use a pre-wired method class** — skips manual wiring:
```python
model = spt.SimCLR(backbone=backbone, projector=projector, temperature=0.5, lr=1e-3)
```

**Enable SLURM requeue + run registry:**
```python
spt.set(cache_dir="/scratch/runs")  # call before Manager()
```

**Add WandB logging:**
```python
from lightning.pytorch.loggers import WandbLogger
trainer = pl.Trainer(..., logger=WandbLogger(project="simclr-cifar10"))
```

**Run from YAML config** (Hydra):
```bash
spt examples/simclr_cifar10_config.yaml
spt examples/simclr_cifar10_config.yaml trainer.max_epochs=1000
```